In [1]:
import torch
import transformers
import peft
import trl

print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.2.1+cu121
4.41.2
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
from huggingface_hub import login
import tqdm as notebook_tqdm

# Paste your HF token here — get from https://huggingface.co/settings/tokens
# Make sure you've accepted the LLaMA 3 license at:
# https://huggingface.co/meta-llama/Meta-Llama-3-8B
HF_TOKEN = "HUGGING_FACE_ACCESS_TOKEN"  # <-- REPLACE THIS

login(token=HF_TOKEN)
print('✅ Logged in')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\himan\.cache\huggingface\token
Login successful
✅ Logged in


In [3]:
dataset_name="teknium/OpenHermes-2.5"
PROMPT_TEMPLATE = """<|system|>
You are a helpful assistant.

<|user|>
{instruction}

{input}

<|assistant|>
{output}"""

model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
from datasets import load_dataset

dataset = load_dataset(dataset_name)
print('✅ Dataset loaded')

✅ Dataset loaded


In [5]:
print(dataset['train'][0])

{'custom_instruction': None, 'topic': None, 'model_name': None, 'model': None, 'skip_prompt_formatting': False, 'category': 'orca', 'conversations': [{'from': 'human', 'value': 'Every day, a tree drops 7 leaves. How many leaves would it drop in a month of February in a non-leap year? Include your logic.', 'weight': None}, {'from': 'gpt', 'value': "Here's the logic behind this:\n\n1. We know that February has 28 days in a non-leap year.\n2. If the tree drops 7 leaves every day, then over the course of February, it would drop:\n   Leaves dropped in February = Leaves per day * Days in February\n   = 7 leaves * 28 days\n   = 196 leaves\n\nSo, the tree would drop 196 leaves in February in a non-leap year.", 'weight': None}], 'views': None, 'language': None, 'id': None, 'title': None, 'idx': None, 'hash': None, 'avatarUrl': None, 'system_prompt': None, 'source': 'airoboros2.2'}


In [6]:
dataset = dataset["train"].shuffle(seed=42).select(range(12000))

# Split into 20k train / 2k test
dataset = dataset.train_test_split(
    test_size=2000,
    seed=42
)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
)

model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(model)
print(tokenizer)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head): Line

In [8]:
def preprocess(example):

    conversations = example["conversations"]

    text = ""

    # Add system prompt
    system_prompt = example.get("system_prompt", "")

    if system_prompt and system_prompt.strip():
        text += f"<|system|>\n{system_prompt}\n\n"
    else:
        text += "<|system|>\nYou are a helpful assistant.\n\n"

    # Process chat messages
    for message in conversations:

        role = message["from"]
        value = message["value"]

        if role == "human":
            text += f"<|user|>\n{value}\n\n"

        elif role == "gpt":
            text += f"<|assistant|>\n{value}\n\n"

    # Add EOS token
    text += tokenizer.eos_token

    return {"text": text}


# Apply preprocessing
dataset = dataset.map(preprocess)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
dataset = dataset.remove_columns(
    [col for col in dataset["train"].column_names if col != "text"]
)

In [10]:
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj"
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
training_args = TrainingArguments(
    output_dir="./tiny_llama_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,

    fp16=True,
    bf16=False,

    logging_steps=25,

    save_steps=200,
    save_total_limit=2,

    report_to="none",

    optim="adamw_torch",
)

# ─────────────────────────────────────────
# STEP 3: Create the trainer
# ─────────────────────────────────────────
trainer = SFTTrainer(
    model=model,

    tokenizer=tokenizer,

    train_dataset=dataset["train"],

    max_seq_length=128,

    dataset_text_field="text",

    args=training_args,
    packing=False
)

d:\transformers\.venv\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
d:\transformers\.venv\Lib\site-packages\trl\trainer\sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
d:\transformers\.venv\Lib\site-packages\trl\trainer\sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [12]:
trainer.train()

  0%|          | 0/625 [00:00<?, ?it/s]

d:\transformers\.venv\Lib\site-packages\transformers\models\llama\modeling_llama.py:649: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 1.6351, 'grad_norm': 0.5191296339035034, 'learning_rate': 0.000192, 'epoch': 0.04}
{'loss': 1.5426, 'grad_norm': 0.5119014978408813, 'learning_rate': 0.00018400000000000003, 'epoch': 0.08}
{'loss': 1.4848, 'grad_norm': 0.5890828967094421, 'learning_rate': 0.00017600000000000002, 'epoch': 0.12}
{'loss': 1.4515, 'grad_norm': 0.6570662260055542, 'learning_rate': 0.000168, 'epoch': 0.16}
{'loss': 1.4138, 'grad_norm': 0.5884917974472046, 'learning_rate': 0.00016, 'epoch': 0.2}
{'loss': 1.464, 'grad_norm': 0.5200597047805786, 'learning_rate': 0.000152, 'epoch': 0.24}
{'loss': 1.4687, 'grad_norm': 0.4795972406864166, 'learning_rate': 0.000144, 'epoch': 0.28}
{'loss': 1.435, 'grad_norm': 0.5097125768661499, 'learning_rate': 0.00013600000000000003, 'epoch': 0.32}


d:\transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'loss': 1.4916, 'grad_norm': 0.5337622165679932, 'learning_rate': 0.00012800000000000002, 'epoch': 0.36}
{'loss': 1.4292, 'grad_norm': 0.5754930377006531, 'learning_rate': 0.00012, 'epoch': 0.4}
{'loss': 1.4361, 'grad_norm': 0.5203669667243958, 'learning_rate': 0.00011200000000000001, 'epoch': 0.44}
{'loss': 1.418, 'grad_norm': 0.5133850574493408, 'learning_rate': 0.00010400000000000001, 'epoch': 0.48}
{'loss': 1.4774, 'grad_norm': 0.5097304582595825, 'learning_rate': 9.6e-05, 'epoch': 0.52}
{'loss': 1.3855, 'grad_norm': 0.5101704001426697, 'learning_rate': 8.800000000000001e-05, 'epoch': 0.56}
{'loss': 1.3868, 'grad_norm': 0.5609750151634216, 'learning_rate': 8e-05, 'epoch': 0.6}
{'loss': 1.399, 'grad_norm': 0.5266794562339783, 'learning_rate': 7.2e-05, 'epoch': 0.64}


d:\transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'loss': 1.4123, 'grad_norm': 0.5607382655143738, 'learning_rate': 6.400000000000001e-05, 'epoch': 0.68}
{'loss': 1.4064, 'grad_norm': 0.558386504650116, 'learning_rate': 5.6000000000000006e-05, 'epoch': 0.72}
{'loss': 1.3847, 'grad_norm': 0.5103675723075867, 'learning_rate': 4.8e-05, 'epoch': 0.76}
{'loss': 1.3951, 'grad_norm': 0.7029467821121216, 'learning_rate': 4e-05, 'epoch': 0.8}
{'loss': 1.3831, 'grad_norm': 0.5323300957679749, 'learning_rate': 3.2000000000000005e-05, 'epoch': 0.84}
{'loss': 1.3973, 'grad_norm': 0.5683994293212891, 'learning_rate': 2.4e-05, 'epoch': 0.88}
{'loss': 1.4045, 'grad_norm': 0.5867022275924683, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.92}
{'loss': 1.4094, 'grad_norm': 0.651599109172821, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.96}


d:\transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'loss': 1.4356, 'grad_norm': 0.5946722626686096, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 21581.9965, 'train_samples_per_second': 0.463, 'train_steps_per_second': 0.029, 'train_loss': 1.437917852783203, 'epoch': 1.0}


TrainOutput(global_step=625, training_loss=1.437917852783203, metrics={'train_runtime': 21581.9965, 'train_samples_per_second': 0.463, 'train_steps_per_second': 0.029, 'total_flos': 7783202674741248.0, 'train_loss': 1.437917852783203, 'epoch': 1.0})

In [13]:
trainer.save_model("./tiny_finetuned_adapters")
tokenizer.save_pretrained("./tiny_finetuned_adapters")

d:\transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('./tiny_finetuned_adapters\\tokenizer_config.json',
 './tiny_finetuned_adapters\\special_tokens_map.json',
 './tiny_finetuned_adapters\\tokenizer.model',
 './tiny_finetuned_adapters\\added_tokens.json')

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load base TinyLlama model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# Attach LoRA adapters
model = PeftModel.from_pretrained(
    base_model,
    "./tiny_finetuned_adapters"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "./tiny_finetuned_adapters"
)

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()

# Save final merged model
merged_model.save_pretrained("./final_tiny_model")

# Save tokenizer
tokenizer.save_pretrained("./final_tiny_model")

print("Merged model saved successfully!")
print("Loaded fine-tuned TinyLlama with adapters!")

Merged model saved successfully!
Loaded fine-tuned TinyLlama with adapters!


In [15]:
from transformers import pipeline

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
merged_model.to(device)

# Create text generation pipeline
pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)


In [19]:
generation_prompt = """### Instruction:
{query}

### Response:
"""

main_prompt = input("Enter your query: ")

generation_prompt = generation_prompt.format(query=main_prompt)

inputs = tokenizer(
    generation_prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Keep only generated answer
response = response.split("### Response:")[-1].strip()

print("\nBot:", response)


Bot: Generative AI is a type of machine learning that uses neural networks to create new content from scratch. The process involves feeding data into a network, which generates new and unexpected output based on the input. It's like creating something out of thin air! Here's how it works: 

The first step is to design your own neural network architecture. This involves defining the layers and connections between them, as well as selecting the appropriate activation function for each layer. Then, you can train the network using a large dataset of labeled examples. Each example will be fed through the network and the resulting output will be used to update the weights of the network.

As the network learns more about the data, it becomes better at generating new examples that are similar but not exactly identical to the original inputs. It's like an algorithm that can generate any number of things with no regard for the specific details or context. This is why generative AI is so intere